# Notebook 03 — Conjunctiva segmentation

**Why:** the models measure the colour of the inner eyelid (palpebral conjunctiva). The datasets come with
that region already cut out by hand, but a live photo from a phone does not. This notebook trains a network
that finds the region automatically in a whole-eye photo.

**Model:** a U-Net. The encoder (the part that "looks") is EfficientNet-B0 pretrained on ImageNet; the decoder
(the part that "draws" the mask) is trained here. Trained on the 218 Eyes-Defy-Anemia eye photos, which come
with hand-drawn masks.

**Honest test:** 5-fold cross-validation, so every photo's mask is predicted by a model that never saw it.
We report Dice and IoU (overlap with the hand-drawn mask; 1.0 = perfect) and save those predicted masks, so the
laptop scripts can check whether the anaemia models still work on automatic masks.

**Outputs** (`/kaggle/working`): `seg_cv_scores.csv`, `pred_masks/*.png`, `conj_seg.onnx` (for the demo app),
`conj_seg.pt`, `seg_examples.png`.

**Run on Kaggle:** dataset `anemia-data`, Accelerator = GPU T4, Internet = On, Save & Run All (about 25–40 min).

In [ ]:
import os, glob, time, random, warnings
import numpy as np, pandas as pd, cv2
import torch, torch.nn as nn, torch.nn.functional as Fn
import timm
try:
    import onnxruntime
except ImportError:                    # not preinstalled on Kaggle; needed only for the final check
    os.system("pip install -q onnx onnxruntime")
from sklearn.model_selection import KFold
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
SEED = 42; random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
QUICK = os.environ.get("ANEMIA_QUICK", "0") == "1"
PRETRAINED = os.environ.get("ANEMIA_PRETRAINED", "1") == "1"
DEV = "cuda" if torch.cuda.is_available() else "cpu"

def find_data():
    hits = glob.glob("/kaggle/input/**/labels.csv", recursive=True)
    if os.environ.get("ANEMIA_DATA"): hits = [os.path.join(os.environ["ANEMIA_DATA"], "labels.csv")] + hits
    assert hits, "labels.csv not found: add the anemia-data dataset"
    return os.path.dirname(hits[0])
DATA = find_data()
OUT = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.environ.get("ANEMIA_OUT", "outputs")
os.makedirs(os.path.join(OUT, "pred_masks"), exist_ok=True)
print(DATA, DEV)

## 1. Data
Photos are resized to 288 × 384 (width × height). Masks are 0/1.

In [ ]:
W, H = 288, 384
lab = pd.read_csv(os.path.join(DATA, "labels.csv"))
lab = lab[lab.mask_path.notna()].reset_index(drop=True)
if QUICK: lab = lab.iloc[:24]
def read_pair(r):
    im = cv2.imread(os.path.join(DATA, r.eye_path)); m = cv2.imread(os.path.join(DATA, r.mask_path), 0)
    return cv2.resize(im, (W, H), interpolation=cv2.INTER_AREA)[:, :, ::-1], (cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST) > 127).astype(np.uint8)
PAIRS = [read_pair(r) for r in lab.itertuples()]
IM = np.stack([p[0] for p in PAIRS]); MK = np.stack([p[1] for p in PAIRS])
print(IM.shape, "mask covers on average", round(MK.mean() * 100, 1), "% of the photo")

## 2. The network
A small U-Net: EfficientNet-B0 encoder (pretrained) + a light decoder with skip connections.

In [ ]:
class UNet(nn.Module):
    def __init__(self, pretrained=True):
        super().__init__()
        self.enc = timm.create_model("efficientnet_b0.ra_in1k", pretrained=pretrained, features_only=True)
        ch = self.enc.feature_info.channels()            # [16, 24, 40, 112, 320]
        dec = [128, 64, 32, 16]
        self.ups = nn.ModuleList()
        prev = ch[-1]
        for skip, out in zip(ch[-2::-1], dec):
            self.ups.append(nn.Sequential(nn.Conv2d(prev + skip, out, 3, padding=1), nn.BatchNorm2d(out), nn.ReLU(inplace=True),
                                          nn.Conv2d(out, out, 3, padding=1), nn.BatchNorm2d(out), nn.ReLU(inplace=True)))
            prev = out
        self.head = nn.Conv2d(prev, 1, 1)
    def forward(self, x):
        f = self.enc(x); h = f[-1]
        for up, skip in zip(self.ups, f[-2::-1]):
            h = Fn.interpolate(h, size=skip.shape[2:], mode="bilinear", align_corners=False)
            h = up(torch.cat([h, skip], 1))
        return Fn.interpolate(self.head(h), size=x.shape[2:], mode="bilinear", align_corners=False)

MEAN = np.array([0.485, 0.456, 0.406], np.float32); STD = np.array([0.229, 0.224, 0.225], np.float32)
def to_t(b): return torch.from_numpy(((b.astype(np.float32) / 255 - MEAN) / STD).transpose(0, 3, 1, 2).copy())

## 3. Training
Loss = binary cross-entropy + Dice loss. Augmentation: horizontal flip, rotation ±15°, zoom ±10%,
and brightness/contrast/colour jitter. (Here colour jitter is fine and useful: the segmenter should find the
eyelid under any lighting. Colour is only measured *after* the mask is drawn.)

In [ ]:
EPOCHS = 2 if QUICK else 40
def aug(im, m, rng):
    if rng.random() < 0.5: im, m = im[:, ::-1], m[:, ::-1]
    M = cv2.getRotationMatrix2D((W / 2, H / 2), rng.uniform(-15, 15), rng.uniform(0.9, 1.1))
    im = cv2.warpAffine(np.ascontiguousarray(im), M, (W, H), borderMode=cv2.BORDER_REFLECT)
    m = cv2.warpAffine(np.ascontiguousarray(m), M, (W, H), flags=cv2.INTER_NEAREST)
    im = im.astype(np.float32) * rng.uniform(0.75, 1.25) + rng.uniform(-20, 20)
    im = im * rng.uniform(0.9, 1.1, size=3)
    return np.clip(im, 0, 255).astype(np.uint8), m

def dice_loss(logit, t):
    p = torch.sigmoid(logit); i = (p * t).sum((1, 2, 3)); return (1 - (2 * i + 1) / (p.sum((1, 2, 3)) + t.sum((1, 2, 3)) + 1)).mean()

def train(idx, seed=SEED):
    torch.manual_seed(seed); rng = np.random.default_rng(seed)
    net = UNet(PRETRAINED).to(DEV)
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    steps = EPOCHS * int(np.ceil(len(idx) / 8)); sch = torch.optim.lr_scheduler.OneCycleLR(opt, 1e-3, total_steps=steps, pct_start=0.1)
    for ep in range(EPOCHS):
        net.train(); order = rng.permutation(idx)
        for i in range(0, len(order), 8):
            b = order[i:i + 8]
            if len(b) < 2: continue
            pr = [aug(IM[k], MK[k], rng) for k in b]
            x = to_t(np.stack([p[0] for p in pr])).to(DEV); t = torch.from_numpy(np.stack([p[1] for p in pr])[:, None].astype(np.float32)).to(DEV)
            out = net(x); loss = Fn.binary_cross_entropy_with_logits(out, t) + dice_loss(out, t)
            opt.zero_grad(); loss.backward(); opt.step(); sch.step()
    return net.eval()

@torch.no_grad()
def predict(net, ims):
    out = []
    for i in range(0, len(ims), 16):
        x = to_t(ims[i:i + 16]).to(DEV)
        p = (torch.sigmoid(net(x)) + torch.sigmoid(net(torch.flip(x, dims=[3]))).flip(dims=[3])) / 2
        out.append(p[:, 0].cpu().numpy())
    return np.concatenate(out)

def keep_largest(m):
    n, cc, st, _ = cv2.connectedComponentsWithStats(m.astype(np.uint8))
    if n <= 1: return m
    return (cc == 1 + np.argmax(st[1:, cv2.CC_STAT_AREA])).astype(np.uint8)

def scores(p, t):
    i = (p & t).sum(); return 2 * i / max(p.sum() + t.sum(), 1), i / max((p | t).sum(), 1)

## 4. Honest test: 5-fold cross-validation

In [ ]:
rows = []; t0 = time.time()
for f, (tr, te) in enumerate(KFold(5, shuffle=True, random_state=SEED).split(IM)):
    net = train(tr, SEED + f); P = predict(net, IM[te])
    for k, p in zip(te, P):
        pm = keep_largest((p > 0.5).astype(np.uint8)); d, j = scores(pm.astype(bool), MK[k].astype(bool))
        r = lab.iloc[k]; rows.append({"subject_id": r.subject_id, "site": r.site, "fold": f, "dice": d, "iou": j})
        full = cv2.imread(os.path.join(DATA, r.mask_path), 0)
        big = cv2.resize(pm * 255, (full.shape[1], full.shape[0]), interpolation=cv2.INTER_NEAREST)
        cv2.imwrite(os.path.join(OUT, "pred_masks", r.subject_id + ".png"), big)
    print(f"fold {f}: mean Dice {np.mean([x['dice'] for x in rows if x['fold'] == f]):.3f}  ({time.time() - t0:.0f}s)", flush=True)
R = pd.DataFrame(rows); R.to_csv(os.path.join(OUT, "seg_cv_scores.csv"), index=False)
print(R.groupby("site")[["dice", "iou"]].describe().round(3).T)
print("overall Dice", R.dice.mean().round(3), "| IoU", R.iou.mean().round(3), "| photos with Dice < 0.5:", int((R.dice < 0.5).sum()))

## 5. Examples (worst, median and best predictions)

In [ ]:
o = R.sort_values("dice"); pick = list(o.index[:3]) + list(o.index[len(o) // 2 - 1:len(o) // 2 + 2]) + list(o.index[-3:])
fig, ax = plt.subplots(3, 3, figsize=(10, 12))
for a, k in zip(ax.flat, pick):
    sid = R.subject_id[k]; i = int(np.where(lab.subject_id == sid)[0][0])
    pm = cv2.resize(cv2.imread(os.path.join(OUT, "pred_masks", sid + ".png"), 0), (W, H)) > 127
    v = IM[i].copy().astype(np.float32); v[MK[i] > 0] = 0.5 * v[MK[i] > 0] + [0, 120, 0]; v[pm] = 0.6 * v[pm] + [100, 0, 100]
    a.imshow(np.clip(v, 0, 255).astype(np.uint8)); a.set_title(f"{sid}  Dice {R.dice[k]:.2f}"); a.axis("off")
plt.suptitle("green = hand-drawn mask, purple = predicted"); plt.tight_layout(); plt.savefig(os.path.join(OUT, "seg_examples.png"), dpi=110)

## 6. Final model on all photos, exported for the demo app
ONNX is a standard model file that runs on a laptop CPU with `onnxruntime`, without PyTorch.

In [ ]:
net = train(np.arange(len(IM)), SEED + 99)
torch.save(net.state_dict(), os.path.join(OUT, "conj_seg.pt"))
dummy = to_t(IM[:1]).to(DEV)
net = net.cpu().eval(); dummy = dummy.cpu()
kw = dict(input_names=["image"], output_names=["logit"], dynamic_axes={"image": {0: "batch"}, "logit": {0: "batch"}}, opset_version=17)
try:
    torch.onnx.export(net, dummy, os.path.join(OUT, "conj_seg.onnx"), dynamo=False, **kw)
except TypeError:                      # older PyTorch has no `dynamo` switch
    torch.onnx.export(net, dummy, os.path.join(OUT, "conj_seg.onnx"), **kw)
import onnxruntime as ort              # check the exported file gives the same answer as PyTorch
ref = net(dummy).detach().numpy(); got = ort.InferenceSession(os.path.join(OUT, "conj_seg.onnx")).run(None, {"image": dummy.numpy()})[0]
print("ONNX matches PyTorch:", np.allclose(ref, got, atol=1e-3))
print("saved conj_seg.onnx; input 1x3x384x288 (ImageNet-normalised RGB), output logit 1x1x384x288")
print(sorted(os.listdir(OUT))[:10])